# Boston Housing — House Price Prediction
## Linear Regression Model

**Author:** Blessing Ebi Uyateide
**Internship:** Codveda Technologies ML Internship 2026
**Level:** 1 | **Task:** 2

### Business Problem
A property valuation firm wants to estimate house prices
based on neighbourhood characteristics. This model helps
buyers understand fair pricing and helps the business
make data-driven valuation decisions.

**Dataset:** Boston Housing Dataset — 506 properties, 13 features
**Target:** Median house value in $1,000s

In [1]:
# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print('Libraries loaded successfully')

Libraries loaded successfully


# Decision Tree Classifier — Boston Housing Price Tier Prediction

We now pivot from regression to a classification task using the same Boston housing dataset. The goal is to predict whether a property falls into a low, medium, or high price tier based on neighborhood features.

In [ ]:
# ADDITIONAL IMPORTS FOR DECISION TREE CLASSIFICATION
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Load the Boston dataset with named columns
column_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS',
    'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV'
]

file_path = r'c:\Users\tinau\OneDrive\文档\Codeva_workspace\4) house Prediction Data Set.csv'
df = pd.read_csv(file_path, header=None, sep=r'\s+', names=column_names)

df.head()

In [ ]:
# Create a categorical target variable for price tier
price_bins = [0, 20, 35, df['MEDV'].max() + 1]
price_labels = ['Low', 'Medium', 'High']
df['price_category'] = pd.cut(df['MEDV'], bins=price_bins, labels=price_labels)

print(df['price_category'].value_counts())

In [ ]:
# Prepare features and target for the decision tree
features = ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'LSTAT']
X = df[features]
y = df['price_category']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train a decision tree classifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

print('Accuracy:', accuracy_score(y_test, y_pred))
print('Macro F1-score:', f1_score(y_test, y_pred, average='macro'))
print('\nClassification report:\n', classification_report(y_test, y_pred))

In [ ]:
# Visualize the trained decision tree structure
plt.figure(figsize=(18, 10))
plot_tree(
    clf,
    feature_names=features,
    class_names=clf.classes_,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Decision Tree for Boston Housing Price Category')
plt.show()

In [ ]:
# Evaluate the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=price_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=price_labels, yticklabels=price_labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Price Category Predictions')
plt.show()

In [ ]:
# Prune the tree to reduce overfitting using cost complexity pruning
path = clf.cost_complexity_pruning_path(X_train_scaled, y_train)
ccp_alphas = path.ccp_alphas
clfs = [
    DecisionTreeClassifier(random_state=42, ccp_alpha=alpha).fit(X_train_scaled, y_train)
    for alpha in ccp_alphas[:-1]
]
train_scores = [model.score(X_train_scaled, y_train) for model in clfs]
test_scores = [model.score(X_test_scaled, y_test) for model in clfs]

plt.figure(figsize=(10, 5))
plt.plot(ccp_alphas[:-1], train_scores, marker='o', label='Train accuracy')
plt.plot(ccp_alphas[:-1], test_scores, marker='o', label='Test accuracy')
plt.xlabel('ccp_alpha')
plt.ylabel('Accuracy')
plt.title('Decision Tree Pruning: Accuracy vs alpha')
plt.legend()
plt.grid(True)
plt.show()

best_alpha = ccp_alphas[:-1][int(np.argmax(test_scores))]
print('Best alpha for pruning:', best_alpha)

pruned_clf = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
pruned_clf.fit(X_train_scaled, y_train)
pred_pruned = pruned_clf.predict(X_test_scaled)

print('Pruned model accuracy:', accuracy_score(y_test, pred_pruned))
print('Pruned model macro F1-score:', f1_score(y_test, pred_pruned, average='macro'))
print('\nPruned classification report:\n', classification_report(y_test, pred_pruned))

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    pruned_clf,
    feature_names=features,
    class_names=pruned_clf.classes_,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title('Pruned Decision Tree for Boston Housing Price Category')
plt.show()

## Summary

- We loaded the Boston housing data and named all columns so each feature is meaningful.
- We converted the continuous median value (`MEDV`) into a categorical target with `Low`, `Medium`, and `High` price tiers.
- A decision tree classifier was trained, visualized, and evaluated with accuracy and F1-score.
- Cost complexity pruning was applied to reduce overfitting, and the pruned tree was evaluated again.